In [30]:
# Подготовка Wazuh-датасета для задачи классификации приоритета 
# Цель ноутбука:
#- объединить все Wazuh JSON-файлы из AIT-ADS
#- выделить только нужные поля
#- сформировать целевой признак `y_priority` на основе `rule_level`
#- сохранить единый датасет для дальнейшего EDA и моделирования

#Важно:
#- в этом ноутбуке `rule_level` используется только для создания target
#- `rule_id` сохраняется в raw-датасете только для контрольного эксперимента leakage
#- AMiner здесь не используется

In [31]:
import json
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
from tqdm import tqdm

In [32]:
from pathlib import Path

# Папка с датасетом
DATA_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/AIT Alert Data Set/8263181/ait_ads")

# Куда сохранять результат
OUTPUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_DIR: /home/chupchik/Рабочий стол/fisrt_stepD/AIT Alert Data Set/8263181/ait_ads
OUTPUT_DIR: /home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset


In [33]:
wazuh_files = sorted(DATA_DIR.glob("*_wazuh.json"))

print(f"Найдено Wazuh-файлов: {len(wazuh_files)}")
for f in wazuh_files:
    print(f.name)

Найдено Wazuh-файлов: 8
fox_wazuh.json
harrison_wazuh.json
russellmitchell_wazuh.json
santos_wazuh.json
shaw_wazuh.json
wardbeck_wazuh.json
wheeler_wazuh.json
wilson_wazuh.json


In [34]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

# =========================
# Пути
# =========================
DATA_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/AIT Alert Data Set/8263181/ait_ads")
OUTPUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

# =========================
# Поиск файлов
# =========================
wazuh_files = sorted(DATA_DIR.glob("*_wazuh.json"))

print(f"Найдено Wazuh-файлов: {len(wazuh_files)}")
for f in wazuh_files:
    print(f.name)

# =========================
# Вспомогательные функции
# =========================
def safe_get(obj, path, default=None):
    cur = obj
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def normalize_rule_groups(groups):
    if groups is None:
        return []
    if isinstance(groups, list):
        return [str(g).strip().lower() for g in groups if pd.notna(g)]
    if isinstance(groups, str):
        return [groups.strip().lower()]
    return []


def make_priority(rule_level):
    if pd.isna(rule_level):
        return None
    try:
        level = int(rule_level)
    except Exception:
        return None

    if level <= 3:
        return "low"
    elif level <= 6:
        return "medium"
    elif level <= 10:
        return "high"
    else:
        return "critical"


# =========================
# Парсер одного Wazuh-файла
# =========================
def parse_wazuh_file(filepath):
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except Exception:
                continue

            row = {
                "scenario": filepath.stem.replace("_wazuh", ""),
                "timestamp": (
                    safe_get(record, ["@timestamp"]) or
                    safe_get(record, ["timestamp"]) or
                    safe_get(record, ["predecoder", "timestamp"])
                ),
                "agent_id": safe_get(record, ["agent", "id"]),
                "agent_name": safe_get(record, ["agent", "name"]),
                "agent_ip": safe_get(record, ["agent", "ip"]),
                "hostname": (
                    safe_get(record, ["predecoder", "hostname"]) or
                    safe_get(record, ["agent", "name"]) or
                    safe_get(record, ["host", "hostname"])
                ),
                "program": (
                    safe_get(record, ["predecoder", "program_name"]) or
                    safe_get(record, ["program_name"]) or
                    safe_get(record, ["decoder", "name"])
                ),
                "location": safe_get(record, ["location"]),
                "full_log": safe_get(record, ["full_log"]),
                "rule_id": safe_get(record, ["rule", "id"]),
                "rule_level": safe_get(record, ["rule", "level"]),
                "rule_description": safe_get(record, ["rule", "description"]),
                "rule_groups": normalize_rule_groups(safe_get(record, ["rule", "groups"])),
            }

            rows.append(row)

    return rows


# =========================
# Чтение всех файлов
# =========================
all_rows = []

for fp in tqdm(wazuh_files, desc="Чтение Wazuh"):
    rows = parse_wazuh_file(fp)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)

print("Размер исходного DataFrame:", df.shape)
print(df.head())


# =========================
# Приведение типов и target
# =========================
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
df["rule_level"] = pd.to_numeric(df["rule_level"], errors="coerce")
df["rule_groups_str"] = df["rule_groups"].apply(lambda x: ",".join(x) if isinstance(x, list) else "")
df["y_priority"] = df["rule_level"].apply(make_priority)

print("\nПосле приведения типов:")
print(df[["timestamp", "rule_level", "y_priority"]].head())


# =========================
# Очистка
# =========================
before_clean = len(df)

df = df[df["rule_level"].notna()].copy()
df = df[df["y_priority"].notna()].copy()
df = df[df["timestamp"].notna()].copy()

after_clean = len(df)

print(f"\nУдалено строк на этапе очистки: {before_clean - after_clean}")
print("После очистки:", df.shape)


# =========================
# Удаление дублей
# =========================
before_dups = len(df)

df = df.drop_duplicates(
    subset=[
        "timestamp",
        "agent_ip",
        "location",
        "full_log",
        "rule_id",
        "rule_level"
    ]
).copy()

after_dups = len(df)

print(f"Удалено дублей: {before_dups - after_dups}")
print("После удаления дублей:", df.shape)


# =========================
# Проверка распределения классов
# =========================
print("\nРаспределение y_priority:")
print(df["y_priority"].value_counts())
print("\nДоли классов:")
print(df["y_priority"].value_counts(normalize=True).round(4))


# =========================
# Проверка утечки через rule_id
# =========================
rule_to_priority = df.groupby("rule_id")["y_priority"].nunique()

single_class_rule_ids = (rule_to_priority == 1).sum()
all_rule_ids = len(rule_to_priority)

print("\nПроверка утечки через rule_id:")
print("rule_id → 1 класс:", single_class_rule_ids)
print("всего rule_id:", all_rule_ids)
if all_rule_ids > 0:
    print("доля:", round(single_class_rule_ids / all_rule_ids, 4))


# =========================
# Итоговый raw-датасет
# =========================
raw_columns = [
    "scenario",
    "timestamp",
    "agent_id",
    "agent_name",
    "agent_ip",
    "hostname",
    "program",
    "location",
    "full_log",
    "rule_id",
    "rule_level",
    "rule_description",
    "rule_groups",
    "rule_groups_str",
    "y_priority"
]

df_raw = df[raw_columns].copy()

print("\nИтоговый raw-датасет:")
print(df_raw.shape)
print(df_raw.head())


# =========================
# Сохранение
# =========================
parquet_path = OUTPUT_DIR / "ait_ads_wazuh_raw.parquet"
csv_sample_path = OUTPUT_DIR / "ait_ads_wazuh_sample.csv"

df_raw.to_parquet(parquet_path, index=False)
df_raw.sample(min(5000, len(df_raw)), random_state=42).to_csv(csv_sample_path, index=False)

print("\nСохранено:")
print(parquet_path)
print(csv_sample_path)

DATA_DIR: /home/chupchik/Рабочий стол/fisrt_stepD/AIT Alert Data Set/8263181/ait_ads
OUTPUT_DIR: /home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset
Найдено Wazuh-файлов: 8
fox_wazuh.json
harrison_wazuh.json
russellmitchell_wazuh.json
santos_wazuh.json
shaw_wazuh.json
wardbeck_wazuh.json
wheeler_wazuh.json
wilson_wazuh.json


Чтение Wazuh: 100%|██████████| 8/8 [00:36<00:00,  4.61s/it]


Размер исходного DataFrame: (2600263, 13)
  scenario                    timestamp agent_id    agent_name  \
0      fox  2022-01-15T02:32:32.000000Z       18  wazuh-client   
1      fox  2022-01-15T02:32:32.000000Z        6  wazuh-client   
2      fox  2022-01-15T02:32:37.000000Z       18  wazuh-client   
3      fox  2022-01-15T02:32:42.000000Z       18  wazuh-client   
4      fox  2022-01-15T02:32:47.000000Z       18  wazuh-client   

          agent_ip         hostname    program         location  \
0    172.17.131.81             mail  freshclam  /var/log/syslog   
1  192.168.128.170  taylorcruz-mail  freshclam  /var/log/syslog   
2    172.17.131.81             mail  freshclam  /var/log/syslog   
3    172.17.131.81             mail  freshclam  /var/log/syslog   
4    172.17.131.81             mail  freshclam  /var/log/syslog   

                                            full_log rule_id  rule_level  \
0  Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...   52507           3   
1  Jan

In [35]:
all_rows = []

for fp in tqdm(wazuh_files, desc="Чтение Wazuh"):
    rows = parse_wazuh_file(fp)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)

print("Размер:", df.shape)
df.head()

Чтение Wazuh: 100%|██████████| 8/8 [00:32<00:00,  4.08s/it]


Размер: (2600263, 13)


,scenario,timestamp,agent_id,agent_name,agent_ip,hostname,program,location,full_log,rule_id,rule_level,rule_description,rule_groups
0,fox,2022-01-15T02:32:32.000000Z,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]"
1,fox,2022-01-15T02:32:32.000000Z,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,/var/log/syslog,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,3,ClamAV database update,"[clamd, freshclam, virus]"
2,fox,2022-01-15T02:32:37.000000Z,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]"
3,fox,2022-01-15T02:32:42.000000Z,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:42 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]"
4,fox,2022-01-15T02:32:47.000000Z,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:47 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]"
